# ⚽ Football Vision — Analisi Tattica su GPU (Colab)

Codice + modelli + analisi su **GPU gratuita**. In output: zip importabile in Scout Lab con
CSV posizioni, statistiche, radar video, dashboard e metriche (atletiche/tattiche).

**Spec Scout Lab / Identity Engine:** il radar deve mostrare i `#Track ID` sopra i giocatori
e produrre una copia web H.264 (`RADARAUTO_*_WEB.mp4`) quando il video originale non e compatibile.
Nel frontend assegnerai identita iniziali e il sistema proporra agganci successivi.

## Come si usa
1. **Runtime → Cambia tipo di runtime → GPU (T4)** → Salva.
2. Esegui le celle in ordine (`Shift+Invio`).
3. Per la clip: **carica un file** (opzione A) — è la più affidabile.
4. Lancia l'analisi e scarica lo zip.
5. Importa lo zip in Scout Lab dal pulsante `Importa zip Colab`.

## 1) Verifica GPU

In [ ]:
import torch
print('✅ GPU attiva:', torch.cuda.get_device_name(0)) if torch.cuda.is_available() else print('❌ GPU NON attiva! Runtime → Cambia tipo di runtime → GPU (T4).')

## 2) Scarica codice + modelli + librerie (~1-2 min)

In [ ]:
!pip -q install ultralytics supervision scikit-learn 2>/dev/null
import os, shutil
if os.path.exists('football-vision'):
    shutil.rmtree('football-vision')
!git clone -q https://github.com/sebavidal2001/football-vision.git
%cd football-vision
import urllib.request
os.makedirs('vista_tattica', exist_ok=True); os.makedirs('clips_input', exist_ok=True)
modelli = {
    'vista_tattica/yolo-football-pitch-detection.pt':
        'https://huggingface.co/martinjolif/yolo-football-pitch-detection/resolve/main/yolo-football-pitch-detection.pt',
    'vista_tattica/giocatori_calcio.pt':
        'https://huggingface.co/uisikdag/yolo-v8-football-players-detection/resolve/main/best.pt',
}
for dst, url in modelli.items():
    if not os.path.exists(dst):
        print('Scarico', os.path.basename(dst), '...'); urllib.request.urlretrieve(url, dst)
print('\n✅ Tutto pronto.')

## 3) Carica la clip (opzione A — consigliata)
Ritaglia prima una clip col programma `AVVIA_ritaglia.bat`, poi caricala qui.
Per una **partita intera** usa l'opzione Google Drive (cella sotto).

In [ ]:
import json, os, shutil, subprocess
from fractions import Fraction


def _run(cmd):
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if res.returncode != 0:
        print(res.stderr[-2000:])
        raise RuntimeError('Comando fallito: ' + ' '.join(cmd))
    return res.stdout


def _probe_video(path):
    data = json.loads(_run([
        'ffprobe', '-v', 'error', '-select_streams', 'v:0',
        '-show_entries', 'stream=codec_name,pix_fmt,avg_frame_rate,r_frame_rate,nb_frames:format=duration',
        '-of', 'json', path
    ]) or '{}')
    stream = (data.get('streams') or [{}])[0]
    duration = float((data.get('format') or {}).get('duration') or 0)
    rate = stream.get('avg_frame_rate') or stream.get('r_frame_rate') or '0/0'
    try:
        fps = float(Fraction(rate))
    except Exception:
        fps = 0.0
    raw_frames = stream.get('nb_frames')
    frames = int(raw_frames) if isinstance(raw_frames, str) and raw_frames.isdigit() else None
    if frames is None and duration > 0 and fps > 0:
        frames = round(duration * fps)
    return {
        'codec': (stream.get('codec_name') or '').lower(),
        'pix_fmt': stream.get('pix_fmt') or 'n/d',
        'duration': duration,
        'fps': fps,
        'frames': frames,
    }


def _fmt_time(seconds):
    seconds = int(round(seconds or 0))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'


def prepara_video(src, dst='clips_input/clip_input.mp4', partita_intera=False):
    if not os.path.exists(src):
        raise FileNotFoundError(f'File non trovato: {src}')

    os.makedirs(os.path.dirname(dst), exist_ok=True)
    info = _probe_video(src)
    fps_txt = f"{info['fps']:.2f}" if info['fps'] else 'n/d'
    frames_txt = f"{info['frames']:,}".replace(',', '.') if info['frames'] else 'n/d'
    print(f"\nVideo sorgente: codec={info['codec'] or 'n/d'}, pixel={info['pix_fmt']}, durata={_fmt_time(info['duration'])}, fps={fps_txt}, frame~{frames_txt}")

    ext = os.path.splitext(src)[1].lower()
    if info['codec'] in ('h264', 'avc1') and ext == '.mp4':
        print('Compatibile: copio il file senza riconversione...')
        shutil.copyfile(src, dst)
    elif info['codec'] in ('h264', 'avc1'):
        print('Video H.264: remux veloce in MP4, senza ricodifica...')
        _run(['ffmpeg', '-y', '-i', src, '-map', '0:v:0', '-c:v', 'copy', '-an', '-movflags', '+faststart', dst])
    else:
        print(f"Codec {info['codec'] or 'sconosciuto'}: riconverto in H.264. Su una partita intera puo volerci molto.")
        _run(['ffmpeg', '-y', '-i', src, '-map', '0:v:0', '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20', '-pix_fmt', 'yuv420p', '-an', '-movflags', '+faststart', dst])

    if not os.path.exists(dst) or os.path.getsize(dst) <= 10000:
        raise RuntimeError('Il video finale non e stato creato correttamente.')

    out = _probe_video(dst)
    out_frames = f"{out['frames']:,}".replace(',', '.') if out['frames'] else 'n/d'
    print(f"\nOK Video pronto: {dst}")
    print(f"   Durata: {_fmt_time(out['duration'])} | fps: {out['fps']:.2f} | frame~{out_frames}")

    if partita_intera and out['duration'] and out['duration'] < 30 * 60:
        print('\nATTENZIONE: durata molto breve per una partita intera.')
        print('Probabilmente e rimasto un file parziale/interrotto. Riesegui questa cella e lascia finire la copia.')


from google.colab import files

os.makedirs('clips_input', exist_ok=True)
up = files.upload()
src = list(up.keys())[0]
prepara_video(src, partita_intera=False)


### Opzione B - Google Drive (per PARTITE INTERE)
Carica il video su Drive, aggiorna `PERCORSO` con il nome esatto e riesegui la cella.

Se il file e gia un MP4 H.264, il notebook fa una copia veloce invece di ricodificare tutta la partita. Alla fine controlla durata e frame: una partita intera non dovrebbe risultare di pochi minuti.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, os, shutil, subprocess
from fractions import Fraction


def _run(cmd):
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if res.returncode != 0:
        print(res.stderr[-2000:])
        raise RuntimeError('Comando fallito: ' + ' '.join(cmd))
    return res.stdout


def _probe_video(path):
    data = json.loads(_run([
        'ffprobe', '-v', 'error', '-select_streams', 'v:0',
        '-show_entries', 'stream=codec_name,pix_fmt,avg_frame_rate,r_frame_rate,nb_frames:format=duration',
        '-of', 'json', path
    ]) or '{}')
    stream = (data.get('streams') or [{}])[0]
    duration = float((data.get('format') or {}).get('duration') or 0)
    rate = stream.get('avg_frame_rate') or stream.get('r_frame_rate') or '0/0'
    try:
        fps = float(Fraction(rate))
    except Exception:
        fps = 0.0
    raw_frames = stream.get('nb_frames')
    frames = int(raw_frames) if isinstance(raw_frames, str) and raw_frames.isdigit() else None
    if frames is None and duration > 0 and fps > 0:
        frames = round(duration * fps)
    return {
        'codec': (stream.get('codec_name') or '').lower(),
        'pix_fmt': stream.get('pix_fmt') or 'n/d',
        'duration': duration,
        'fps': fps,
        'frames': frames,
    }


def _fmt_time(seconds):
    seconds = int(round(seconds or 0))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'


def prepara_video(src, dst='clips_input/clip_input.mp4', partita_intera=False):
    if not os.path.exists(src):
        raise FileNotFoundError(f'File non trovato: {src}')

    os.makedirs(os.path.dirname(dst), exist_ok=True)
    info = _probe_video(src)
    fps_txt = f"{info['fps']:.2f}" if info['fps'] else 'n/d'
    frames_txt = f"{info['frames']:,}".replace(',', '.') if info['frames'] else 'n/d'
    print(f"\nVideo sorgente: codec={info['codec'] or 'n/d'}, pixel={info['pix_fmt']}, durata={_fmt_time(info['duration'])}, fps={fps_txt}, frame~{frames_txt}")

    ext = os.path.splitext(src)[1].lower()
    if info['codec'] in ('h264', 'avc1') and ext == '.mp4':
        print('Compatibile: copio il file senza riconversione...')
        shutil.copyfile(src, dst)
    elif info['codec'] in ('h264', 'avc1'):
        print('Video H.264: remux veloce in MP4, senza ricodifica...')
        _run(['ffmpeg', '-y', '-i', src, '-map', '0:v:0', '-c:v', 'copy', '-an', '-movflags', '+faststart', dst])
    else:
        print(f"Codec {info['codec'] or 'sconosciuto'}: riconverto in H.264. Su una partita intera puo volerci molto.")
        _run(['ffmpeg', '-y', '-i', src, '-map', '0:v:0', '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20', '-pix_fmt', 'yuv420p', '-an', '-movflags', '+faststart', dst])

    if not os.path.exists(dst) or os.path.getsize(dst) <= 10000:
        raise RuntimeError('Il video finale non e stato creato correttamente.')

    out = _probe_video(dst)
    out_frames = f"{out['frames']:,}".replace(',', '.') if out['frames'] else 'n/d'
    print(f"\nOK Video pronto: {dst}")
    print(f"   Durata: {_fmt_time(out['duration'])} | fps: {out['fps']:.2f} | frame~{out_frames}")

    if partita_intera and out['duration'] and out['duration'] < 30 * 60:
        print('\nATTENZIONE: durata molto breve per una partita intera.')
        print('Probabilmente e rimasto un file parziale/interrotto. Riesegui questa cella e lascia finire la copia.')


# Mostra i file presenti in MyDrive, per trovare il nome esatto del video.
print('\nFile/cartelle in MyDrive:')
for f in sorted(os.listdir('/content/drive/MyDrive')):
    print('   ', f)

# Scrivi qui il nome ESATTO del tuo video, poi riesegui la cella.
PERCORSO = '/content/drive/MyDrive/partita.mp4'   # es. /content/drive/MyDrive/PSG_Inter.mp4
# Se e dentro una cartella: /content/drive/MyDrive/Calcio/PSG_Inter.mp4

if not os.path.exists(PERCORSO):
    print('\nFile non trovato:', PERCORSO, '- controlla il nome nella lista qui sopra.')
else:
    prepara_video(PERCORSO, partita_intera=True)


## 4) Analisi su GPU
**Clip breve (1-5 min):** `SALTO=2`, `OGNI_CAMPO=1`, `SOLO_DATI=False`.

**PARTITA INTERA:** `SALTO=3`, `OGNI_CAMPO=2`, **`SOLO_DATI=True`** (niente video radar →
più veloce e leggero). Stima su GPU T4: **~45-60 min** una partita intera.

In [ ]:
SALTO = 2
OGNI_CAMPO = 1
SOLO_DATI = False   # True per PARTITA INTERA (niente video radar, solo report)

import os
video = 'clips_input/clip_input.mp4'
base = os.path.splitext(os.path.basename(video))[0]
csv_pos = f'output/POSIZIONIAUTO_{base}.csv'
novideo = '--no_video' if SOLO_DATI else ''

print('▶ 1/5 Radar/dati + rilevamento giocatori...')
!python vista_tattica/genera_radar_auto.py "{video}" --salto {SALTO} --ogni_campo {OGNI_CAMPO} --imgsz 1280 {novideo}
print('\n▶ 2/5 Heatmap + statistiche...')
!python analisi/stats_giocatori.py "{csv_pos}" --min_rilevazioni 20
print('\n▶ 3/5 Dashboard di confronto...')
!python analisi/confronto_giocatori.py "output/STATISTICHE_{base}.csv"
print('\n▶ 4/5 Metriche scouting (formazione, andamento, atletico)...')
!python analisi/metriche_avanzate.py "{csv_pos}" --min_rilevazioni 20
print('\n▶ 5/5 Report PDF unico...')
!python analisi/report_pdf.py "{base}" --dir output
print('\nScout Lab output attesi:')
for f in [
    f'output/POSIZIONIAUTO_{base}.csv',
    f'output/STATISTICHE_{base}.csv',
    f'output/RADARAUTO_{base}.mp4',
    f'output/RADARAUTO_{base}_WEB.mp4',
    f'output/DASHBOARD_{base}.png',
]:
    print(('OK ' if os.path.exists(f) else '-- '), f)
print('\n✅ FATTO. Importa report.zip in Scout Lab: i Track ID saranno usati per Identity Engine.')

## 5) Anteprima risultati (il PDF completo è nello zip)

In [ ]:
from IPython.display import Image, display
import glob
for pattern, titolo in [('output/FORMAZIONE_*.png', '=== FORMAZIONE / MODULO ==='),
                        ('output/ANDAMENTO_*.png', '=== ANDAMENTO TATTICO ==='),
                        ('output/REPORT_*.png', '=== REPORT SCOUTING ==='),
                        ('output/DASHBOARD_*.png', '=== CONFRONTO GIOCATORI ===')]:
    for f in glob.glob(pattern):
        print(titolo); display(Image(f))

## 6) Scarica lo zip per Scout Lab
Importa questo zip nel frontend con `Importa zip Colab`. Deve contenere `POSIZIONIAUTO_*`, `STATISTICHE_*` e, se `SOLO_DATI=False`, `RADARAUTO_*_WEB.mp4`.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('report', 'zip', 'output')
files.download('report.zip')